In [ ]:
import numpy as np
from numba import cuda
from numba import config
import cv2
import matplotlib.pyplot as plt
from google.colab import files
import os, math, shutil, glob

config.CUDA_ENABLE_PYNVJITLINK = 1  # resuelve bug de versiones


## Utilidades compartidas

In [ ]:
# ─── Carga de imagen individual ───────────────────────────────────────────────
def upload_image():
    """Solicita al usuario subir una imagen y la retorna como array BGR."""
    uploaded = files.upload()
    for filename, data in uploaded.items():
        print(f'Imagen subida: "{filename}"')
        img = cv2.imdecode(np.frombuffer(data, np.uint8), cv2.IMREAD_COLOR)
        return filename, img
    return None, None

def show_images(images, titles, cols=2, cmap_list=None):
    """Muestra una lista de imágenes con sus títulos."""
    n = len(images)
    rows = math.ceil(n / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(7 * cols, 5 * rows))
    axes = np.array(axes).flatten()
    for ax, img, title, i in zip(axes, images, titles, range(n)):
        cmap = (cmap_list[i] if cmap_list else None)
        if img.ndim == 3:
            ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        else:
            ax.imshow(img, cmap=cmap or 'gray')
        ax.set_title(title)
        ax.axis('off')
    for ax in axes[n:]:
        ax.axis('off')
    plt.tight_layout()
    plt.show()

# ─── Utilidades de vídeo ──────────────────────────────────────────────────────
FRAMES_DIR   = 'frames'          # frames extraídos del vídeo original
RESULTS_DIR  = 'resultados'      # carpeta raíz de vídeos con filtros

def setup_video_dirs(filter_name):
    """Crea/limpia carpeta de frames y la subcarpeta de resultados para un filtro."""
    os.makedirs(FRAMES_DIR, exist_ok=True)
    out_dir = os.path.join(RESULTS_DIR, filter_name)
    os.makedirs(out_dir, exist_ok=True)
    return out_dir

def extract_frames(video_path, max_frames=None):
    """
    Extrae frames del vídeo y los guarda en FRAMES_DIR.
    Devuelve (fps, width, height, total_extraídos).
    """
    # Limpiar frames previos
    if os.path.exists(FRAMES_DIR):
        shutil.rmtree(FRAMES_DIR)
    os.makedirs(FRAMES_DIR)

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise IOError(f'No se pudo abrir el vídeo: {video_path}')

    fps     = cap.get(cv2.CAP_PROP_FPS)
    width   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height  = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total   = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    limit   = min(total, max_frames) if max_frames else total
    print(f'Vídeo: {total} frames | {fps:.2f} FPS | {width}×{height}')

    count = 0
    while count < limit:
        ret, frame = cap.read()
        if not ret:
            break
        cv2.imwrite(os.path.join(FRAMES_DIR, f'frame_{count:05d}.jpg'), frame)
        count += 1
    cap.release()
    print(f'Extraídos {count} frames en "{FRAMES_DIR}".')
    return fps, width, height, count

def build_video(frames_dir, output_path, fps, width, height):
    """Une los frames procesados en un vídeo MP4."""
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    writer = cv2.VideoWriter(output_path, fourcc, fps, (width, height))
    frame_files = sorted(glob.glob(os.path.join(frames_dir, 'frame_*.jpg')))
    for fp in frame_files:
        frame = cv2.imread(fp)
        if frame is not None:
            writer.write(frame)
    writer.release()
    print(f'Vídeo guardado en: {output_path}')

def apply_filter_to_video(filter_name, process_fn, fps, width, height):
    """
    Pipeline genérico: lee cada frame de FRAMES_DIR,
    aplica process_fn(frame) -> frame_procesado,
    guarda en resultados/<filter_name>/ y construye el vídeo final.
    """
    out_dir = setup_video_dirs(filter_name)
    frame_files = sorted(glob.glob(os.path.join(FRAMES_DIR, 'frame_*.jpg')))
    total = len(frame_files)
    print(f'[{filter_name}] Procesando {total} frames...')

    for i, fp in enumerate(frame_files):
        frame = cv2.imread(fp)
        if frame is None:
            continue
        processed = process_fn(frame)
        cv2.imwrite(os.path.join(out_dir, os.path.basename(fp)), processed)
        if (i + 1) % 100 == 0 or (i + 1) == total:
            print(f'  {i + 1}/{total} frames procesados.')

    video_path = os.path.join(RESULTS_DIR, f'{filter_name}.mp4')
    build_video(out_dir, video_path, fps, width, height)
    return video_path

print('Utilidades cargadas correctamente.')


Utilidades cargadas correctamente.


## Kernels CUDA

In [ ]:
# ─── 1. Invertir colores ─────────────────────────────────────────────────────
@cuda.jit
def invert_colors_kernel(input_image, output_image):
    idx = cuda.grid(1)
    if idx < input_image.size:
        output_image[idx] = 255 - input_image[idx]

# ─── 2. Blur promedio ─────────────────────────────────────────────────────────
@cuda.jit
def blur_image_kernel(input_image, output_image, height, width, channels, kernel_size):
    idx = cuda.grid(1)
    if idx >= input_image.size:
        return
    h = idx // (width * channels)
    w = (idx // channels) % width
    c = idx % channels
    radius = kernel_size // 2
    sum_val = 0.0
    count   = 0
    for dh in range(-radius, radius + 1):
        for dw in range(-radius, radius + 1):
            nh, nw = h + dh, w + dw
            if 0 <= nh < height and 0 <= nw < width:
                sum_val += input_image[(nh * width * channels) + (nw * channels) + c]
                count   += 1
    output_image[idx] = int(sum_val / count) if count > 0 else input_image[idx]

# ─── 3. Recorte ───────────────────────────────────────────────────────────────
@cuda.jit
def crop_image_kernel(input_image, output_image, orig_h, orig_w, channels,
                      x_start, y_start, w_crop, h_crop):
    idx_out = cuda.grid(1)
    if idx_out >= output_image.size:
        return
    h_out = idx_out // (w_crop * channels)
    w_out = (idx_out // channels) % w_crop
    c     = idx_out % channels
    h_in  = y_start + h_out
    w_in  = x_start + w_out
    if 0 <= h_in < orig_h and 0 <= w_in < orig_w:
        output_image[idx_out] = input_image[(h_in * orig_w * channels) + (w_in * channels) + c]
    else:
        output_image[idx_out] = 0

# ─── 4. Umbralización ─────────────────────────────────────────────────────────
@cuda.jit
def umbral_rgb_kernel(img, salida, T):
    x, y = cuda.grid(2)
    if x < img.shape[0] and y < img.shape[1]:
        promedio = (img[x, y, 2] + img[x, y, 1] + img[x, y, 0]) / 3.0
        salida[x, y] = np.uint8(255) if promedio > T else np.uint8(0)

# ─── 5. Sobel ─────────────────────────────────────────────────────────────────
@cuda.jit
def sobel_kernel(input_image, output_image, height, width):
    x, y = cuda.grid(2)
    if x <= 0 or x >= height - 1 or y <= 0 or y >= width - 1:
        return
    Kx = ((-1, 0, 1), (-2, 0, 2), (-1, 0, 1))
    Ky = ((-1, -2, -1), (0, 0, 0), (1, 2, 1))
    Gx, Gy = 0.0, 0.0
    for i in range(3):
        for j in range(3):
            px = input_image[x + i - 1, y + j - 1]
            Gx += px * Kx[i][j]
            Gy += px * Ky[i][j]
    mag = math.sqrt(Gx * Gx + Gy * Gy)
    output_image[x, y] = int(min(mag, 255))

print('Kernels CUDA compilados.')


Kernels CUDA compilados.


## Funciones de filtro (imagen individual)

In [ ]:
TPB_1D = 256   # Threads per block para kernels 1D
TPB_2D = (16, 16)  # Threads per block para kernels 2D

def apply_invert(img):
    """Invierte los colores de una imagen BGR."""
    flat = img.flatten()
    d_in  = cuda.to_device(flat)
    d_out = cuda.device_array_like(d_in)
    blocks = (flat.size + TPB_1D - 1) // TPB_1D
    invert_colors_kernel[blocks, TPB_1D](d_in, d_out)
    return d_out.copy_to_host().reshape(img.shape).astype(np.uint8)

def apply_blur(img, kernel_size=7):
    """Aplica desenfoque por promedio con tamaño de kernel configurable."""
    h, w, c = img.shape
    flat  = img.flatten()
    d_in  = cuda.to_device(flat)
    d_out = cuda.device_array_like(d_in)
    blocks = (flat.size + TPB_1D - 1) // TPB_1D
    blur_image_kernel[blocks, TPB_1D](d_in, d_out, h, w, c, kernel_size)
    return d_out.copy_to_host().reshape(img.shape).astype(np.uint8)

def apply_crop(img, x_start=100, y_start=50, w_crop=400, h_crop=300):
    """Recorta una región de la imagen."""
    orig_h, orig_w, c = img.shape
    w_crop = min(w_crop, orig_w - x_start)
    h_crop = min(h_crop, orig_h - y_start)
    flat   = img.flatten()
    size   = w_crop * h_crop * c
    d_in   = cuda.to_device(flat)
    d_out  = cuda.device_array(shape=size, dtype=flat.dtype)
    blocks = (size + TPB_1D - 1) // TPB_1D
    crop_image_kernel[blocks, TPB_1D](d_in, d_out, orig_h, orig_w, c,
                                       x_start, y_start, w_crop, h_crop)
    return d_out.copy_to_host().reshape((h_crop, w_crop, c)).astype(np.uint8)

def apply_threshold(img, T=164.0):
    """Umbralización: devuelve imagen binaria (grayscale)."""
    img_f = img.astype(np.float32)
    h, w, _ = img.shape
    d_in  = cuda.to_device(img_f)
    d_out = cuda.device_array((h, w), dtype=np.uint8)
    bx = (h + TPB_2D[0] - 1) // TPB_2D[0]
    by = (w + TPB_2D[1] - 1) // TPB_2D[1]
    umbral_rgb_kernel[(bx, by), TPB_2D](d_in, d_out, T)
    cuda.synchronize()
    return d_out.copy_to_host()

def apply_sobel(img):
    """Detección de bordes Sobel sobre imagen en escala de grises."""
    gray  = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY).astype(np.uint8)
    h, w  = gray.shape
    d_in  = cuda.to_device(gray)
    d_out = cuda.device_array_like(gray)
    bx = math.ceil(h / TPB_2D[0])
    by = math.ceil(w / TPB_2D[1])
    sobel_kernel[(bx, by), TPB_2D](d_in, d_out, h, w)
    cuda.synchronize()
    result = d_out.copy_to_host()
    return cv2.normalize(result, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

print('Funciones de filtro listas.')


Funciones de filtro listas.


### Ejercicio 1 – Invertir colores

In [ ]:
_, img = upload_image()
if img is not None:
    result = apply_invert(img)
    show_images([img, result], ['Original', 'Invertida'])


Saving WhatsApp Video 2026-03-11 at 09.53.09.mp4 to WhatsApp Video 2026-03-11 at 09.53.09.mp4
Imagen subida: "WhatsApp Video 2026-03-11 at 09.53.09.mp4"


### Ejercicio 2 – Blur promedio

In [ ]:
_, img = upload_image()
if img is not None:
    result = apply_blur(img, kernel_size=7)
    show_images([img, result], ['Original', f'Blur 7×7'])


### Ejercicio 3 – Recorte

In [ ]:
_, img = upload_image()
if img is not None:
    result = apply_crop(img, x_start=100, y_start=50, w_crop=400, h_crop=300)
    show_images([img, result], ['Original', 'Recortada'])


### Ejercicio 4 – Umbralización

In [ ]:
_, img = upload_image()
if img is not None:
    result = apply_threshold(img, T=164.0)
    show_images([img, result], ['Original', 'Umbral (T=164)'])


### Ejercicio 5 – Detección de bordes (Sobel)

In [ ]:
_, img = upload_image()
if img is not None:
    result = apply_sobel(img)
    show_images([img, result], ['Original', 'Sobel'])


# Procesamiento de Vídeo con CUDA

## Paso 1 – Cargar vídeo y extraer frames

Sube un vídeo. Se extraerán todos sus frames (o solo los primeros `max_frames`)
y se guardarán en la carpeta **`frames/`**.


In [ ]:
print('Sube un vídeo (.mp4, .avi, …):')
uploaded = files.upload()
video_filename = list(uploaded.keys())[0] if uploaded else None

MAX_FRAMES = None   # ← Cambia a un número (p.ej. 300) para limitar los frames

if video_filename:
    video_fps, video_w, video_h, n_frames = extract_frames(video_filename, MAX_FRAMES)
else:
    print('No se subió ningún vídeo.')


Sube un vídeo (.mp4, .avi, …):


Saving WhatsApp Video 2026-03-11 at 09.53.09.mp4 to WhatsApp Video 2026-03-11 at 09.53.09 (1).mp4
Vídeo: 1033 frames | 30.00 FPS | 576×576
Extraídos 1033 frames en "frames".


## Paso 2 – Aplicar filtros y reconstruir vídeos

Por cada filtro se recorren los frames de `frames/`, se aplica el filtro en GPU,
se guardan las imágenes en `resultados/<filtro>/` y se unen para crear el vídeo final.


In [ ]:
# Wrappers para vídeo: todos deben devolver una imagen BGR
# Los filtros que NO cambian dimensiones devuelven (frame, None, None)
# El filtro de recorte devuelve (frame, nuevo_w, nuevo_h) para que el VideoWriter se ajuste

def video_invert(frame):
    return apply_invert(frame)

def video_blur(frame):
    return apply_blur(frame, kernel_size=7)

def video_threshold(frame):
    gray = apply_threshold(frame, T=164.0)
    return cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)

def video_sobel(frame):
    edges = apply_sobel(frame)
    return cv2.cvtColor(edges, cv2.COLOR_GRAY2BGR)

# ─── Parámetros de recorte (ajústalos según tu vídeo) ────────────────────────
CROP_X      = 100   # columna de inicio
CROP_Y      = 50    # fila de inicio
CROP_WIDTH  = 400   # ancho de la región recortada
CROP_HEIGHT = 300   # alto  de la región recortada

def video_crop(frame):
    return apply_crop(frame,
                      x_start=CROP_X, y_start=CROP_Y,
                      w_crop=CROP_WIDTH, h_crop=CROP_HEIGHT)

# ─── Pipeline de vídeo genérico (soporta cambio de dimensiones) ──────────────
def apply_filter_to_video_v2(filter_name, process_fn, fps, orig_w, orig_h):
    """
    Igual que apply_filter_to_video pero detecta automáticamente las dimensiones
    del primer frame procesado, permitiendo filtros que cambian el tamaño.
    """
    out_dir = setup_video_dirs(filter_name)
    frame_files = sorted(glob.glob(os.path.join(FRAMES_DIR, 'frame_*.jpg')))
    total = len(frame_files)
    print(f'[{filter_name}] Procesando {total} frames...')

    out_w, out_h = orig_w, orig_h  # se actualizará con el primer frame
    writer = None

    for i, fp in enumerate(frame_files):
        frame = cv2.imread(fp)
        if frame is None:
            continue
        processed = process_fn(frame)

        # Inicializar VideoWriter con las dimensiones reales del primer frame procesado
        if writer is None:
            out_h, out_w = processed.shape[:2]
            fourcc = cv2.VideoWriter_fourcc(*'mp4v')
            video_path = os.path.join(RESULTS_DIR, f'{filter_name}.mp4')
            writer = cv2.VideoWriter(video_path, fourcc, fps, (out_w, out_h))
            print(f'  Dimensiones de salida: {out_w}×{out_h}')

        cv2.imwrite(os.path.join(out_dir, os.path.basename(fp)), processed)
        writer.write(processed)

        if (i + 1) % 100 == 0 or (i + 1) == total:
            print(f'  {i + 1}/{total} frames procesados.')

    if writer:
        writer.release()
        print(f'✔  {filter_name} → {video_path}')
    return video_path

# ─── Diccionario de filtros ───────────────────────────────────────────────────
FILTROS = {
    'invertir': video_invert,
    'blur':     video_blur,
    'recorte':  video_crop,
    'umbral':   video_threshold,
    'sobel':    video_sobel,
}

os.makedirs(RESULTS_DIR, exist_ok=True)

for nombre, fn in FILTROS.items():
    apply_filter_to_video_v2(nombre, fn, video_fps, video_w, video_h)
    print()

print('¡Procesamiento de vídeo completado!')
print(f'Los vídeos están en la carpeta "{RESULTS_DIR}/".')


[invertir] Procesando 1033 frames...
  Dimensiones de salida: 576×576
  100/1033 frames procesados.
  200/1033 frames procesados.
  300/1033 frames procesados.
  400/1033 frames procesados.
  500/1033 frames procesados.
  600/1033 frames procesados.
  700/1033 frames procesados.
  800/1033 frames procesados.
  900/1033 frames procesados.
  1000/1033 frames procesados.
  1033/1033 frames procesados.
✔  invertir → resultados/invertir.mp4

[blur] Procesando 1033 frames...
  Dimensiones de salida: 576×576
  100/1033 frames procesados.
  200/1033 frames procesados.
  300/1033 frames procesados.
  400/1033 frames procesados.
  500/1033 frames procesados.
  600/1033 frames procesados.
  700/1033 frames procesados.
  800/1033 frames procesados.
  900/1033 frames procesados.
  1000/1033 frames procesados.
  1033/1033 frames procesados.
✔  blur → resultados/blur.mp4

[recorte] Procesando 1033 frames...
  Dimensiones de salida: 400×300
  100/1033 frames procesados.
  200/1033 frames procesados.
 

## Paso 3 – Descargar los vídeos resultantes


In [ ]:
for nombre in FILTROS:
    path = os.path.join(RESULTS_DIR, f'{nombre}.mp4')
    if os.path.exists(path):
        print(f'Descargando {path}…')
        files.download(path)


Descargando resultados/invertir.mp4…


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Descargando resultados/blur.mp4…


<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Descargando resultados/sobel.mp4…


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>